In [ ]:
#@title Install Dependencies
%%capture
!pip install gradio openai -q

In [ ]:
#@title Set Your OpenAI API Key
import os
from getpass import getpass

# Enter your OpenAI API key when prompted
api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key
print("API key set successfully!")

In [ ]:
#@title LLM Judge Comparator App
"""
LLM Judge Comparator - Compare different judge personalities
Shows how judge configuration affects evaluation results
"""

import gradio as gr
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

JUDGE_PROFILES = {
    "Strict Judge": {
        "system_prompt": """You are a strict, demanding evaluator. You have high standards and rarely give perfect scores. 
You focus on finding flaws and areas for improvement. A score of 5 should be exceptional and rare.
Be specific about what's missing or could be better.""",
        "temperature": 0.2
    },
    "Balanced Judge": {
        "system_prompt": """You are a fair, balanced evaluator. You acknowledge both strengths and weaknesses equally.
You give credit where it's due but also point out areas for improvement.
Use the full 1-5 scale appropriately.""",
        "temperature": 0.5
    },
    "Lenient Judge": {
        "system_prompt": """You are an encouraging, supportive evaluator. You focus on what the response does well.
You give the benefit of the doubt and emphasize positive aspects.
You believe in constructive feedback that motivates improvement.""",
        "temperature": 0.7
    }
}

SAMPLE_OUTPUTS = {
    "Email Response (Good)": {
        "prompt": "Write a professional email declining a meeting request",
        "output": """Subject: Re: Meeting Request

Hi Sarah,

Thank you for reaching out about scheduling a meeting. Unfortunately, I have a conflict during the proposed time and won't be able to attend.

Would any of these alternatives work for you?
- Thursday 2-3pm
- Friday 10-11am

If neither works, please let me know your availability and I'll do my best to accommodate.

Best regards,
Alex"""
    },
    "Email Response (Mediocre)": {
        "prompt": "Write a professional email declining a meeting request",
        "output": """Hi,

Can't make the meeting. Maybe another time.

Thanks"""
    },
    "Technical Summary (Good)": {
        "prompt": "Summarize the benefits of microservices architecture",
        "output": """Microservices architecture offers several key benefits:

1. **Independent Deployment**: Teams can deploy services without coordinating with others
2. **Technology Flexibility**: Each service can use the best tool for its job
3. **Fault Isolation**: One service failing doesn't bring down the whole system
4. **Scalability**: Scale only the services that need it

However, it adds complexity in service communication and data consistency. Best suited for large teams with complex domains."""
    },
    "Technical Summary (Mediocre)": {
        "prompt": "Summarize the benefits of microservices architecture",
        "output": "Microservices are good because they let you break up your app into smaller pieces. This makes things easier to manage. Big companies like Netflix use them."
    }
}


def evaluate_with_judge(prompt, output, judge_type):
    """Evaluate output using specified judge personality"""
    
    if not os.environ.get("OPENAI_API_KEY"):
        return "Error: Please set your OpenAI API key in the cell above."
    
    profile = JUDGE_PROFILES[judge_type]
    
    eval_prompt = f"""Evaluate this AI output on a 1-5 scale for: Accuracy, Clarity, Helpfulness, Professionalism.

ORIGINAL PROMPT: {prompt}

OUTPUT TO EVALUATE:
{output}

Provide:
1. Score for each criterion (1-5)
2. Overall score
3. Brief explanation (2-3 sentences)

Format with clear headers."""

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": profile["system_prompt"]},
                {"role": "user", "content": eval_prompt}
            ],
            temperature=profile["temperature"],
            max_tokens=500
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"


def compare_all_judges(prompt, output):
    """Run evaluation with all three judges"""
    results = {}
    for judge_type in JUDGE_PROFILES.keys():
        results[judge_type] = evaluate_with_judge(prompt, output, judge_type)
    return results["Strict Judge"], results["Balanced Judge"], results["Lenient Judge"]


def load_sample(sample_name):
    if sample_name in SAMPLE_OUTPUTS:
        sample = SAMPLE_OUTPUTS[sample_name]
        return sample["prompt"], sample["output"]
    return "", ""


# Build Gradio interface
with gr.Blocks(title="LLM Judge Comparator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # LLM Judge Comparator
    
    See how different judge "personalities" evaluate the same output.
    
    **For Product Managers:** This shows why judge calibration matters.
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Input")
            
            sample_dropdown = gr.Dropdown(
                choices=list(SAMPLE_OUTPUTS.keys()),
                label="Load Sample",
                value="Email Response (Good)"
            )
            
            prompt_input = gr.Textbox(label="Original Prompt", lines=2)
            output_input = gr.Textbox(label="AI Output", lines=6)
            
            compare_btn = gr.Button("Compare All Judges", variant="primary")
            
            gr.Markdown("""
            ### Judge Profiles
            
            | Judge | Style | Temperature |
            |-------|-------|-------------|
            | Strict | Demanding, finds flaws | 0.2 |
            | Balanced | Fair, objective | 0.5 |
            | Lenient | Encouraging, positive | 0.7 |
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("### Judge Evaluations")
            
            with gr.Tab("Strict Judge"):
                strict_output = gr.Markdown()
            
            with gr.Tab("Balanced Judge"):
                balanced_output = gr.Markdown()
            
            with gr.Tab("Lenient Judge"):
                lenient_output = gr.Markdown()
    
    gr.Markdown("""
    ---
    ### PM Insight: Judge Calibration
    
    **Why scores differ:**
    - System prompt shapes evaluation style
    - Temperature affects consistency vs creativity
    - Same rubric, different interpretation
    
    **Best practice:** Use multiple judges and analyze disagreements.
    Consistent scores across judges = high confidence.
    """)
    
    # Event handlers
    sample_dropdown.change(
        fn=load_sample,
        inputs=[sample_dropdown],
        outputs=[prompt_input, output_input]
    )
    
    compare_btn.click(
        fn=compare_all_judges,
        inputs=[prompt_input, output_input],
        outputs=[strict_output, balanced_output, lenient_output]
    )
    
    demo.load(
        fn=load_sample,
        inputs=[sample_dropdown],
        outputs=[prompt_input, output_input]
    )

In [ ]:
#@title Launch App - Copy the gradio.live URL below
demo.launch(share=True)